<center><p float="center">
  <img src="https://vectorseek.com/wp-content/uploads/2023/08/Massachusetts-Institute-of-Technology-MIT-Logo-Vector.svg-.png" width="360"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200"/>
</p></center>

<center><font size=10><b>AI and Data Science</b></font></center>
<center><font size=6><b>Project: Full Code</b></font></center>

# **Problem Statement**

## Business Context

In last-mile logistics, roughly 10% of all shipments encounter delivery exceptions: failed attempts, address mismatches, damaged packages, refused deliveries, and weather delays. Each failed delivery adds direct reattempt costs, and the downstream impact is far steeper: consumers stop shopping with a retailer after just 2 to 3 failed deliveries, and supply chain disruptions cost companies millions of dollars per year.

Despite these costs, most organizations still handle exception triage and resolution manually. Operations staff read through status logs and messy driver notes, cross-reference customer profiles and internal playbooks, decide on a resolution action, and draft customer notifications, all under time pressure. Teams rely on support tickets, tribal knowledge, and rigid rule engines that cannot handle the nuanced, multi-factor judgment calls that real exceptions demand. A VIP customer with a perishable package and a broken gate code on a second attempt needs a different resolution than a standard customer's first failed attempt on a non-perishable item, yet manual processes treat both with the same slow, inconsistent workflow.

Key KPIs affected include:
- Exception resolution time (constrained by manual triage throughput)
- Escalation accuracy (missed or unnecessary supervisor escalations)
- Customer communication quality and personalization
- Cost per exception (reattempt costs, spoilage write-offs, unnecessary truck rolls)
- Customer retention and lifetime value

If unaddressed, logistics providers face rising exception-handling costs, inconsistent service quality, preventable customer churn, and loss of competitive ground to players already investing in AI-powered exception detection and response automation.

An AI-powered multi-agent system can ingest delivery logs, cross-reference customer profiles and locker availability, retrieve resolution policy from an operational playbook, and generate validated decisions with personalized customer communications. With it, logistics providers can automate the full exception-handling pipeline from detection through resolution, with built-in quality validation and supervisor escalation where policy requires it.

## Objective

The objective is to build a proof of concept (POC) of an AI-powered multi-agent delivery exception handling system for the last-mile delivery operations of a mid-sized retailer. The system should:
- use the operator's available data to process exceptions end to end;
- ingest raw delivery status logs, including noisy, duplicated, and multi-row shipment events, and correctly separate actionable exceptions from routine operational noise;
- decide the appropriate resolution action (reschedule, reroute to locker, replace, or return to sender) by reasoning over playbook rules, customer context, package constraints, and locker eligibility, while following operational policies on perishable handling, fragile thresholds, and locker capacity;
- escalate to a human supervisor when policy requires it, and avoid unnecessary escalations that waste supervisor capacity;
- generate personalized customer notifications with tone, channel, and content matched to customer tier and exception severity; and
- produce auditable decisions with step-by-step rationale, validation traces, and evaluation metrics across five dimensions: task completion rate, escalation accuracy, tool call accuracy, reasoning trajectory coherence, and end-to-end latency.

The end goal is to demonstrate measurable accuracy and consistency on curated exception scenarios, enough to justify scaling the approach to real-time operations with live delivery feeds, broader exception types, and multi-region deployment.

## Data Dictionary

The data comprises two types of files: CSV files and a PDF document.

### CSV Files

**Customer Profiles (`customers.csv`)** - 12 rows

| Attribute | Type | Values / Description |
|:---|:---|:---|
| customer_id | string | Unique ID (`CUST-001` to `CUST-012`), links to `customer_id` in the delivery logs |
| name | string | Customer full name (**PII**) |
| tier | string | STANDARD, PREMIUM, VIP |
| preferred_channel | string | SMS, EMAIL |
| exceptions_last_90d | integer | Range: 0-6 |
| active_credit | float | Range: \$0.00-\$20.00 |

**Pickup Lockers (`lockers.csv`)** - 6 rows

| Attribute | Type | Values / Description |
|:---|:---|:---|
| locker_id | string | Unique ID (`LOC-001` to `LOC-006`) |
| address | string | Locker street address |
| zip_code | string | 10001-10006 |
| capacity_status | string | AVAILABLE, LIMITED, FULL |
| operating_hours | string | e.g., "6AM-10PM" or "24 hours" |
| max_package_size | string | SMALL, MEDIUM, LARGE |

**Delivery Status Logs (`delivery_logs.csv`)** - 13 rows, 10 unique shipments

| Attribute | Type | Values / Description |
|:---|:---|:---|
| shipment_id | string | `SHP-001` to `SHP-010`. Multiple rows may share the same ID for multi-event shipments. |
| timestamp | datetime | ISO format |
| status_code | string | One of seven codes: DELIVERED, ATTEMPTED, DAMAGED, ADDRESS_ISSUE, REFUSED, WEATHER_DELAY, SCANNED |
| status_description | string | Free-text driver/handler notes |
| customer_id | string | Links to `customers.csv` (`CUST-001` to `CUST-012`) |
| delivery_address | string | Full address including zip code |
| package_type | string | STANDARD, PERISHABLE, FRAGILE |
| package_size | string | SMALL, MEDIUM, LARGE |
| attempt_number | integer | 0 for pre-delivery scans, 1+ for delivery attempts |
| is_duplicate_scan | boolean | True for repeated scan events |

**Ground Truth Labels (`ground_truth.csv`)** - 13 rows, aligned 1:1 with delivery logs

| Attribute | Type | Values / Description |
|:---|:---|:---|
| shipment_id | string | Matches delivery logs (`SHP-001` to `SHP-010`) |
| is_exception | string | YES, NO |
| expected_resolution | string | RESCHEDULE, REPLACE, REROUTE_TO_LOCKER, RETURN_TO_SENDER, N/A |
| expected_tone | string | FORMAL, CASUAL, N/A |
| should_escalate | string | YES, NO, N/A |
| ground_truth_reasoning | string | Step-by-step justification tracing each label to specific playbook rules, customer attributes, and locker constraints |

### Exception Resolution Playbook (`exception_resolution_playbook.pdf`)

An internal operations manual for human dispatchers, written in natural language. It covers exception handling procedures, the escalation matrix, locker reroute eligibility rules, and customer communication guidelines.

# **Please read the instructions carefully before starting the project.**

This is a Python Notebook file with blocks of code pre-filled in alignment with a possible solution for the business use case at hand, and some blank sections where the code has to be written from scratch.

* Please feel free to
    * leverage the pre-filled code blocks as they are, or
    * update the pre-filled code blocks to incorporate necessary changes as per your desired solution workflow for the business problem at hand, or
    * discard the pre-filled code blocks and write the entire code from scratch
* Notebook sections and code blocks that contain instructions and tasks to be performed are mentioned.
* Blanks '\_\_\_\_\_' are provided in the notebook that need to be filled with an appropriate code to get the correct result.
    * With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis under the respective notebook sections as per the grading rubric requirements.

# **Installing and Importing Necessary Libraries and Dependencies**

In [1]:
# Install the latest compatible versions
!uv pip install -q openai langchain langchain-openai langchain-text-splitters langgraph langchain-huggingface sentence-transformers langchain-chroma chromadb pypdf

**Note:**

- After running the install cell above, restart the runtime (for Google Colab) or the notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- You might see a warning about package dependencies. You can ignore it: the command above installs all the libraries and dependencies this notebook needs.

**Optional: check the installed versions**

Run the cell below to see which version of each library got installed. To use a specific version of a library, install it like this:

`!pip install -q library-name==version`

For example: `!pip install -q langgraph==1.2.16`

In [2]:
# Optional: check the installed versions
!pip show openai langchain langchain-openai langchain-text-splitters langgraph langchain-huggingface sentence-transformers langchain-chroma chromadb pypdf

Name: openai
Version: 2.54.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: langchain-openai
---
Name: langchain
Version: 1.4.8
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-openai
Version: 1.8.5
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/openai
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: certifi, langchain-core, openai, tiktoken
Required-by: 
---
Name:

In [3]:
# write the code to import all necessary libraries
# Note: the pre-filled code uses os, json, csv, time, operator, pandas (pd), typing (Literal, Annotated, TypedDict),
# collections.defaultdict, pydantic (BaseModel, Field), langchain_openai.ChatOpenAI, langchain_core (SystemMessage,
# HumanMessage, Document, tool), langchain_text_splitters.RecursiveCharacterTextSplitter, langchain_huggingface.HuggingFaceEmbeddings,
# langchain_chroma.Chroma, pypdf.PdfReader, langgraph.graph (StateGraph, START, END), and IPython.display (Image, display)
import os
import json
import csv
import time
import operator
from typing import Literal, Annotated, TypedDict
from collections import defaultdict

import pandas as pd
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pypdf import PdfReader
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# **LLM Setup**

We load the GL OpenAI access token from Google Colab Secrets (saved as `openai_api_key`) and point the OpenAI client to the Great Learning proxy.

In [10]:
import os                                                                       # Access environment variables
from google.colab import userdata                                               # Read secrets stored in Colab

# Store OpenAI credentials in environment variables
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')                   # Set the GL OpenAI access token
os.environ['OPENAI_BASE_URL'] = "https://aibe.mygreatlearning.com/openai/v1"    # Set the Great Learning proxy URL

For the problem at hand, we use two LLM instances with separate responsibilities. Both use GPT-5.6 Luna, the fastest and most cost-efficient model in OpenAI's GPT-5.6 family:
- a generation model with `reasoning_effort="none"`, for fast, low-cost responses from the Resolution and Communication Agents, and
- a validation model with `reasoning_effort="medium"`, which reasons more carefully before the Critic Agent accepts, revises, or escalates a decision and before scoring reasoning coherence.

GPT-5.6 models are reasoning models, so they take a `reasoning_effort` setting in place of `temperature`.

In [11]:
# write the code to set up the LLM(s)
# Use gpt-5.6-luna through the Great Learning proxy. GPT-5.6 models take a reasoning_effort setting (none, low, medium, high) in place of temperature.
# Note: the pre-filled agents use two LLMs named gen_llm (generation) and eval_llm (validation)

gen_llm = ChatOpenAI(
    model="gpt-5.6-luna",
    reasoning_effort="none",
)

eval_llm = ChatOpenAI(
    model="gpt-5.6-luna",
    reasoning_effort="medium",
)

# **Data Loading and Vector Database Setup**

In [12]:
# write the code to define the file paths for all five dataset files
CUSTOMERS_PATH = "customers.csv"
LOCKERS_PATH = "lockers.csv"
DELIVERY_LOGS_PATH = "delivery_logs.csv"
GROUND_TRUTH_PATH = "ground_truth.csv"
PLAYBOOK_PDF_PATH = "exception_resolution_playbook.pdf"


## Loading the CSV files

In [13]:
# write the code to load delivery_logs.csv
# Note: store its file path in DELIVERY_LOGS_PATH; the read_delivery_logs tool uses it
# DELIVERY_LOGS_PATH is already defined above and read directly by read_delivery_logs()
delivery_logs_df = pd.read_csv(DELIVERY_LOGS_PATH)
print(f"Loaded delivery_logs.csv with shape: {delivery_logs_df.shape}")

Loaded delivery_logs.csv with shape: (13, 10)


In [14]:
# write the code to load ground_truth.csv into a DataFrame named ground_truth_df
# Note: pandas reads the text "N/A" as a missing value by default; use keep_default_na=False to keep the labels as text
ground_truth_df = pd.read_csv(GROUND_TRUTH_PATH, keep_default_na=False)
print(f"Loaded ground_truth.csv with shape: {ground_truth_df.shape}")

Loaded ground_truth.csv with shape: (13, 6)


## Loading the customer and locker data

In [15]:
# write the code to load customers.csv and lockers.csv into DataFrames named customers_df and lockers_df
# Note: the customer and locker tools use these names; read zip_code as text, e.g. pd.read_csv(..., dtype={"zip_code": str})
customers_df = pd.read_csv(CUSTOMERS_PATH)
lockers_df = pd.read_csv(LOCKERS_PATH, dtype={"zip_code": str})

Let's quickly verify that both files loaded correctly by checking their shape and a few sample rows.

In [16]:
# write the code to check the shape and a few sample rows of both DataFrames
print("customers_df shape:", customers_df.shape)
display(customers_df.head(3))

print("lockers_df shape:", lockers_df.shape)
display(lockers_df.head(3))

customers_df shape: (12, 6)


,customer_id,name,tier,preferred_channel,exceptions_last_90d,active_credit
0,CUST-001,John Smith,VIP,EMAIL,5,15.0
1,CUST-002,Jane Doe,PREMIUM,EMAIL,2,0.0
2,CUST-003,Robert Johnson,STANDARD,SMS,0,0.0


lockers_df shape: (6, 6)


,locker_id,address,zip_code,capacity_status,operating_hours,max_package_size
0,LOC-001,"150 Main Street, Greenfield",10001,AVAILABLE,6AM-10PM,LARGE
1,LOC-002,"42 Oak Avenue, Riverside",10002,LIMITED,7AM-9PM,MEDIUM
2,LOC-003,"88 Elm Boulevard, Westdale",10003,FULL,24 hours,LARGE


## Loading the PDF file

In [17]:
# write the code to read the playbook PDF into a list of LangChain Documents named playbook_pages
# Note: store each page's number in the Document's metadata as "page"; the search_playbook tool uses it for citations
reader = PdfReader(PLAYBOOK_PDF_PATH)
playbook_pages = [
    Document(
        page_content=page.extract_text() or "",
        metadata={"page": idx + 1, "source": PLAYBOOK_PDF_PATH},
    )
    for idx, page in enumerate(reader.pages)
]
print(f"Loaded {len(playbook_pages)} playbook pages.")

Loaded 10 playbook pages.


## Creating Vector Database

The extracted PDF content will now be split into smaller overlapping chunks for vector search.

In [18]:
# write the code to split playbook_pages into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)
playbook_chunks = text_splitter.split_documents(playbook_pages)
print(f"Split into {len(playbook_chunks)} chunks.")

Split into 36 chunks.


Next, the embedding model is initialized using HuggingFace.

In [19]:
# write the code to initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The playbook chunks will now be embedded and stored in a ChromaDB vector store.

A retriever will be configured to return the top-k most relevant chunks per query.

In [20]:
# write the code to store the chunks in a Chroma vector store and create a retriever
# Note: the search_playbook tool uses a retriever named playbook_retriever
vectorstore = Chroma.from_documents(
    documents=playbook_chunks,
    embedding=embeddings,
    collection_name="exception_resolution_playbook",
)
playbook_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# **Defining Tools**

Let's define the tools to be used by the multi-agent system.

## Delivery Log Reader

This tool reads all delivery log rows from the CSV file into a list of dictionaries.

In [21]:
@tool
def read_delivery_logs() -> list[dict]:
    """Read all delivery log rows from the CSV file."""
    # Open the CSV and parse each row into a dictionary keyed by column headers
    with open(DELIVERY_LOGS_PATH, "r") as f:
        return list(csv.DictReader(f))

## Customer Profile Lookup

Both tools below read from the customer and locker data loaded earlier.

This tool fetches a customer profile from the customer data by customer ID.

- The `include_pii` flag controls whether the customer's name is included in the response.
- When set to `False` (default), the name field is removed before returning.
- This enforces data privacy at the tool level: only the Communication Agent receives the customer's name; the Resolution and Critic Agents receive the profile without it.

In [22]:
@tool
def lookup_customer_profile(customer_id: str, include_pii: bool = False) -> dict:
    """Fetch a customer profile by customer ID. PII (name) only included when explicitly requested."""
    # Find the customer record by ID
    match = customers_df[customers_df["customer_id"] == customer_id]

    # Return empty dict if customer not found
    if match.empty:
        return {}

    # Convert the matching row to a dictionary
    profile = match.to_dict(orient="records")[0]

    # Redact customer name unless PII is explicitly requested
    if not include_pii:
        profile.pop("name", None)

    return profile

## Locker Availability Checker

This tool finds lockers in the same zip code as the delivery address, then evaluates each locker's eligibility based on three constraints:
- package size must fit within the locker's maximum capacity,
- the locker cannot be full, and
- lockers with limited capacity only accept small packages.

Each locker is returned with an eligible flag and a human-readable reason explaining the decision.

In [23]:
@tool
def check_locker_availability(zip_code: str, package_size: str) -> list[dict]:
    """Find compatible lockers in the same zip code. Returns eligibility with reasoning."""
    # Map package sizes to numeric levels for comparison
    size_hierarchy = {"SMALL": 1, "MEDIUM": 2, "LARGE": 3}
    pkg_level = size_hierarchy.get(package_size, 0)

    # Find lockers matching the delivery zip code
    lockers = lockers_df[lockers_df["zip_code"] == zip_code].to_dict(orient="records")

    results = []
    for locker in lockers:
        locker_max = size_hierarchy.get(locker["max_package_size"], 0)

        # Check constraint 1: package must fit within locker's max size
        if locker_max < pkg_level:
            locker["eligible"] = False
            locker["reason"] = f"Locker max {locker['max_package_size']} < package {package_size}"

        # Check constraint 2: locker must not be full
        elif locker["capacity_status"] == "FULL":
            locker["eligible"] = False
            locker["reason"] = "Locker is FULL"

        # Check constraint 3: limited lockers only accept small packages
        elif locker["capacity_status"] == "LIMITED" and package_size != "SMALL":
            locker["eligible"] = False
            locker["reason"] = "Locker is LIMITED - only SMALL packages accepted"

        # All constraints passed
        else:
            locker["eligible"] = True
            locker["reason"] = "Compatible"

        results.append(locker)

    return results

## Playbook Search

This tool searches the playbook vector store for chunks most relevant to the given query, and returns each chunk's text content along with its source page number.

- This metadata is used for document citation tracking in the evaluation output.

In [24]:
@tool
def search_playbook(query: str) -> list[dict]:
    """Retrieve relevant playbook sections via vector search. Returns chunks with page metadata."""
    # Run similarity search against the ChromaDB vector store
    docs = playbook_retriever.invoke(query)

    # Return chunk content and page number for each retrieved document
    return [
        {"content": d.page_content, "page": d.metadata.get("page", "?")}
        for d in docs
    ]

## Escalation Rule Engine

This tool defines a fully deterministic escalation rule engine that evaluates hard-coded business rules without any LLM involvement.

It checks two categories of triggers:
- automatic triggers (third delivery attempt, VIP with high exception history, damaged perishables, perishable weather delays exceeding 4 hours, fraud-indicating addresses), and
- discretionary triggers (standard customers with unusually high exception counts, premium perishable weather delays).

Delay hours are parsed from free-text descriptions using regex to handle varied formats like "5hr", "5 hours", or "5.5hr".

In [25]:
@tool
def check_escalation_rules(customer_tier: str, exceptions_last_90d: int,
                            attempt_number: int, package_type: str,
                            status_code: str, status_description: str) -> dict:
    """Deterministic escalation rule engine. Evaluates hard-coded business rules."""
    import re
    triggers = []

    # --- Automatic triggers ---

    # 3rd failed attempt triggers escalation regardless of customer tier
    if attempt_number >= 3:
        triggers.append("AUTOMATIC: 3rd failed delivery attempt")

    # VIP customers with 3+ exceptions in last 90 days
    if customer_tier == "VIP" and exceptions_last_90d >= 3:
        triggers.append(f"AUTOMATIC: VIP customer with {exceptions_last_90d} exceptions in 90d (>=3)")

    # Any damaged perishable package needs fast replacement decision
    if status_code == "DAMAGED" and package_type == "PERISHABLE":
        triggers.append("AUTOMATIC: Damaged perishable package")

    # Perishable with weather delay over 4 hours - item likely compromised
    if status_code == "WEATHER_DELAY" and package_type == "PERISHABLE":
        # Extract delay hours from free-text using regex (handles "5hr", "5 hours", "5.5hr")
        hour_matches = re.findall(r'(\d+(?:\.\d+)?)\s*(?:hr|hour|hours)', status_description.lower())
        if hour_matches:
            hours = float(hour_matches[0])
            if hours > 4:
                triggers.append(f"AUTOMATIC: Perishable with {hours}hr delay (>4hr threshold)")

    # Address issues suggesting potential fraud
    fraud_keywords = ["vacant", "demolished", "construction site", "empty lot"]
    if status_code == "ADDRESS_ISSUE" and any(kw in status_description.lower() for kw in fraud_keywords):
        triggers.append("AUTOMATIC: Potential fraud - address is vacant/demolished")

    # --- Discretionary triggers ---

    # Standard customer with unusually high exception count signals systemic issue
    if customer_tier == "STANDARD" and exceptions_last_90d > 5:
        triggers.append(f"DISCRETIONARY: Standard customer with {exceptions_last_90d} exceptions in 90d (>5)")

    # Premium customer with perishable in weather delay - borderline urgency
    if customer_tier == "PREMIUM" and package_type == "PERISHABLE" and status_code == "WEATHER_DELAY":
        triggers.append("DISCRETIONARY: Premium customer with perishable in weather delay")

    return {
        "has_triggers": len(triggers) > 0,                                       # Quick boolean check for any triggers
        "trigger_count": len(triggers),                                          # Total number of triggers found
        "triggers": triggers                                                     # Detailed list of trigger descriptions
    }

# **Multi-Agent Architecture**

Our system processes one shipment at a time through six nodes. Two run plain Python code, three are LLM agents, and a supervisor decides the order.

| Node | Type | Role |
|:---|:---|:---|
| Preprocessor | Code | Removes duplicate scans, merges multi-row shipments, filters routine noise, and gathers context with the tools |
| Supervisor | Code | Decides which agent runs next and when a case needs escalation |
| Resolution Agent | LLM (generation model) | Decides whether the event is an exception and which resolution action to take |
| Critic Agent | LLM (validation model) | Reviews the decision and returns ACCEPT, REVISE, or ESCALATE |
| Communication Agent | LLM (generation model) | Writes the customer notification |
| Finalize | Code | Packages the final output and records latency |

This design combines two multi-agent patterns:
- a **supervisor pattern**: every agent reports back to the supervisor, which routes the work, and
- a **sequential pattern**: an exception flows from the Resolution Agent to the Critic Agent to the Communication Agent, with the Critic able to send a decision back for revision.

## Multi-Agent State

The state is the shared record that every node reads from and writes to.

- Each node returns only the fields it changes, and LangGraph merges them into the state.
- The two log fields use `Annotated[list[str], operator.add]`, so LangGraph appends new entries instead of replacing the list. This builds a complete audit trail across all nodes.
- The redacted `customer_profile` goes to the Resolution and Critic Agents; only the Communication Agent reads `customer_profile_full`, which includes the customer's name.

In [26]:
class AgentState(TypedDict):
    """Shared state passed between all nodes in the workflow."""
    # Input
    shipment_id: str
    raw_rows: list[dict]                                   # Raw delivery log rows for this shipment

    # Set by the preprocessor
    consolidated_event: dict                               # One clean event per shipment
    is_noise: bool                                         # True for routine events that need no action
    customer_profile: dict                                 # Profile without the customer's name
    customer_profile_full: dict                            # Profile with the name, for the Communication Agent only
    locker_availability: list[dict]                        # Lockers in the delivery zip code, with eligibility
    playbook_context: list[dict]                           # Retrieved playbook chunks with page numbers
    escalation_signals: dict                               # Output of the escalation rule engine

    # Set by the agents
    resolution_output: dict                                # {is_exception, resolution, rationale}
    critic_output: dict                                    # {decision, rationale}
    critic_feedback: str                                   # Critic's reason when it asks for a revision
    revision_count: int                                    # Number of revisions so far
    communication_output: dict                             # {tone_label, communication_message}

    # Set by the supervisor
    next_agent: str                                        # Which node runs next
    escalated: bool                                        # Whether the case goes to a human supervisor

    # Audit trail and timing
    tool_calls_log: Annotated[list[str], operator.add]     # Every tool and agent call
    trajectory_log: Annotated[list[str], operator.add]     # Every decision, in order
    start_time: float
    latency_sec: float
    final_output: dict

## Preprocessor

The preprocessor prepares each shipment before any agent sees it. We first define four helper functions:
- `deduplicate_rows` removes repeated scan events,
- `consolidate_event` merges multi-row shipments into one event, keeping driver notes from earlier attempts,
- `is_routine_noise` flags routine status codes that need no action, and
- `fetch_context` calls the four context tools.

In [27]:
def deduplicate_rows(raw_rows: list[dict]) -> list[dict]:
    """Remove duplicate scan events."""
    return [r for r in raw_rows if r.get("is_duplicate_scan", "False") != "True"]

In [28]:
def consolidate_event(unique_rows: list[dict], raw_rows: list[dict]) -> dict:
    """Merge a multi-row shipment into one event, using the row with the highest attempt number."""
    primary = max(unique_rows, key=lambda r: int(r["attempt_number"])) if unique_rows else raw_rows[0]

    # Keep driver notes from earlier attempts, since they may explain the current problem
    prior_notes = [
        f"Attempt {r['attempt_number']}: {r['status_description']}"
        for r in unique_rows if r is not primary
    ]

    return {
        "shipment_id": primary["shipment_id"],
        "timestamp": primary["timestamp"],
        "status_code": primary["status_code"],
        "status_description": primary["status_description"],
        "customer_id": primary["customer_id"],
        "delivery_address": primary["delivery_address"],
        "package_type": primary["package_type"],
        "package_size": primary["package_size"],
        "attempt_number": int(primary["attempt_number"]),
        "prior_attempt_notes": prior_notes
    }

In [29]:
def is_routine_noise(event: dict) -> bool:
    """Flag routine status codes whose driver notes show no sign of a problem."""
    routine_codes = {"DELIVERED", "IN_TRANSIT", "OUT_FOR_DELIVERY", "SCANNED"}
    if event["status_code"] not in routine_codes:
        return False

    # Words that suggest a routine-looking event still needs attention
    anomaly_words = ["damage", "wrong", "suspicious", "overdue", "missing", "unexpected", "misroute",
                     "lost", "stolen", "abandoned", "leak", "crush", "broke", "delay", "late", "fraud"]
    description = event["status_description"].lower()
    return not any(word in description for word in anomaly_words)

In [30]:
def fetch_context(event: dict) -> tuple[dict, list[str]]:
    """Gather everything the agents need by calling the four context tools."""
    customer_id = event["customer_id"]
    zip_code = event["delivery_address"].split(",")[-1].strip()

    customer_profile = lookup_customer_profile.invoke({"customer_id": customer_id, "include_pii": False})
    customer_profile_full = lookup_customer_profile.invoke({"customer_id": customer_id, "include_pii": True})
    locker_availability = check_locker_availability.invoke({"zip_code": zip_code, "package_size": event["package_size"]})

    query = f"{event['status_code']} {event['package_type']} {event['status_description'][:100]}"
    playbook_context = search_playbook.invoke({"query": query})

    escalation_signals = check_escalation_rules.invoke({
        "customer_tier": customer_profile.get("tier", "STANDARD"),
        "exceptions_last_90d": customer_profile.get("exceptions_last_90d", 0),
        "attempt_number": event["attempt_number"],
        "package_type": event["package_type"],
        "status_code": event["status_code"],
        "status_description": event["status_description"]
    })

    context = {
        "customer_profile": customer_profile,
        "customer_profile_full": customer_profile_full,
        "locker_availability": locker_availability,
        "playbook_context": playbook_context,
        "escalation_signals": escalation_signals
    }
    tool_log = [
        f"TOOL: lookup_customer_profile({customer_id}, pii=False)",
        f"TOOL: lookup_customer_profile({customer_id}, pii=True)",
        f"TOOL: check_locker_availability({zip_code}, {event['package_size']})",
        "TOOL: search_playbook(query)",
        "TOOL: check_escalation_rules(...)"
    ]
    return context, tool_log

The preprocessor node runs these steps in order. Routine noise skips the tools and agents entirely, which saves time and LLM cost.

In [31]:
def preprocessor_node(state: AgentState) -> dict:
    """Clean the delivery logs, filter routine noise, and gather context for the agents."""
    start = time.time()
    unique_rows = deduplicate_rows(state["raw_rows"])
    event = consolidate_event(unique_rows, state["raw_rows"])
    trajectory = [f"preprocessor: {len(state['raw_rows'])} raw rows -> {len(unique_rows)} after dedup"]

    # Routine noise: record the outcome directly, with no tools or agents
    if is_routine_noise(event):
        trajectory.append(f"preprocessor: {event['status_code']} is routine noise, skipping tools and agents")
        return {
            "consolidated_event": event,
            "is_noise": True,
            "resolution_output": {"is_exception": "NO", "resolution": "N/A",
                                  "rationale": "Routine status with no sign of a problem."},
            "start_time": start,
            "trajectory_log": trajectory
        }

    # Exceptions and unclear cases: gather context for the agents
    context, tool_log = fetch_context(event)
    return {"consolidated_event": event, "is_noise": False, **context,
            "start_time": start, "tool_calls_log": tool_log, "trajectory_log": trajectory}

## Supervisor

The supervisor runs after the preprocessor and after every agent. It reads the state and applies five rules in order:

1. **Routine noise** goes straight to finalize.
2. **No decision yet:** send the case to the Resolution Agent.
3. **Decision not reviewed yet:** send it to the Critic Agent.
4. **Critic asked for a revision, and the limit isn't reached:** send it back to the Resolution Agent with the Critic's feedback.
5. **Review complete:** set the escalation flag, then send exceptions to the Communication Agent (once) and finish.

A case is escalated to a human supervisor when the escalation rule engine finds an AUTOMATIC trigger, when the Critic returns ESCALATE, or when the Critic still disagrees after the revision limit. Automatic triggers are enforced in code, so the LLM can't overrule them.

In [32]:
# write the code to define the supervisor node, named supervisor_node, that applies the five routing rules above
# Notes:
# - define MAX_REVISIONS, the maximum number of times the Critic can send a decision back
# - return a dict with "next_agent" set to one of: "resolution_agent", "critic_agent", "communication_agent", "finalize"
# - when the Critic returns REVISE under the limit, also clear resolution_output and critic_output so both agents run again
# - set "escalated" to True for exceptions with an AUTOMATIC trigger, a Critic ESCALATE, or a REVISE at the limit
# - add entries to trajectory_log as a list, e.g. {"trajectory_log": ["supervisor: ..."]}
MAX_REVISIONS = 2


def supervisor_node(state: AgentState) -> dict:
    """Route shipments between preprocessor, agents, escalation, and finalize."""
    # Rule 1: Routine noise goes straight to finalize
    if state.get("is_noise", False):
        return {
            "next_agent": "finalize",
            "escalated": False,
            "trajectory_log": ["supervisor: routine noise -> finalize"],
        }

    resolution_out = state.get("resolution_output") or {}
    critic_out = state.get("critic_output") or {}
    revision_count = state.get("revision_count", 0)

    # Rule 2: No resolution decision yet -> send to Resolution Agent
    if not resolution_out:
        return {
            "next_agent": "resolution_agent",
            "trajectory_log": ["supervisor: no resolution yet -> resolution_agent"],
        }

    # Rule 3: Resolution exists, but not reviewed by Critic yet -> send to Critic Agent
    if not critic_out:
        return {
            "next_agent": "critic_agent",
            "trajectory_log": ["supervisor: decision unreviewed -> critic_agent"],
        }

    critic_decision = critic_out.get("decision")

    # Rule 4: Critic requested REVISE and revision limit has not been reached
    if critic_decision == "REVISE" and revision_count <= MAX_REVISIONS:
        return {
            "next_agent": "resolution_agent",
            "resolution_output": {},
            "critic_output": {},
            "trajectory_log": [
                f"supervisor: critic requested REVISE (count={revision_count}/{MAX_REVISIONS}) -> resolution_agent"
            ],
        }

    # Rule 5: Review complete -> compute escalation flag and route exceptions to Communication Agent once
    has_auto_trigger = any(
        str(t).startswith("AUTOMATIC:")
        for t in (state.get("escalation_signals") or {}).get("triggers", [])
    )
    critic_escalated = critic_decision == "ESCALATE"
    revise_at_limit = critic_decision == "REVISE" and revision_count > MAX_REVISIONS

    is_exception = resolution_out.get("is_exception") == "YES"
    escalated = bool(is_exception and (has_auto_trigger or critic_escalated or revise_at_limit))

    comm_already_ran = bool(state.get("communication_output"))
    if is_exception and not comm_already_ran:
        return {
            "next_agent": "communication_agent",
            "escalated": escalated,
            "trajectory_log": [
                f"supervisor: review complete (escalated={escalated}) -> communication_agent"
            ],
        }

    return {
        "next_agent": "finalize",
        "escalated": escalated,
        "trajectory_log": [
            f"supervisor: review complete (escalated={escalated}) -> finalize"
        ],
    }

## Resolution Agent

We first define the output schema for the Resolution Agent. Structured output makes the agent return exactly these three fields.

In [33]:
class ResolutionOutput(BaseModel):
    """Resolution Agent output schema."""
    is_exception: Literal["YES", "NO"] = Field(description="Whether this delivery event is a real actionable exception")
    resolution: Literal["RESCHEDULE", "REROUTE_TO_LOCKER", "REPLACE", "RETURN_TO_SENDER", "N/A"] = Field(
        description="Resolution action. N/A if is_exception is NO")
    rationale: str = Field(description="Step-by-step reasoning for the classification and resolution decision")

Next, we define the Resolution Agent's system prompt, which sets the agent's role, classification rules, resolution guidelines, and consistency rules.
- The `{critic_feedback}` placeholder is filled with the Critic's feedback when the agent revises a decision.

In [34]:
RESOLUTION_AGENT_SYSTEM_PROMPT = """You are the Resolution Agent for a last-mile delivery operations multi-agent system.
Your job is to inspect a consolidated shipment event along with customer context, locker eligibility, escalation signals, and operational playbook excerpts, then determine:
1. `is_exception`: "YES" if the event represents an actionable delivery exception requiring operational handling, or "NO" if it is non-actionable / routine.
2. `resolution`: One of "RESCHEDULE", "REROUTE_TO_LOCKER", "REPLACE", "RETURN_TO_SENDER", or "N/A" (use "N/A" only when `is_exception` is "NO").
3. `rationale`: Clear, step-by-step reasoning citing the status code, driver notes/prior attempt notes, package constraints (`package_type`, `package_size`), customer context, locker eligibility, escalation signals, and playbook rules.

Follow these guidelines:
- Apply playbook rules strictly, including perishable handling, fragile package handling, refused deliveries, address issues, repeat delivery attempts, and locker eligibility (`eligible == True`).
- Only select `REROUTE_TO_LOCKER` when at least one locker in `LOCKER AVAILABILITY` has `"eligible": true` and aligns with the playbook.
- Keep `is_exception` and `resolution` consistent ("N/A" when `is_exception` is "NO"; a valid action when `is_exception` is "YES").
{critic_feedback}"""

# keep {critic_feedback} at the end: the Critic's feedback is inserted there on a revision

Next, we define a helper function that formats retrieved playbook chunks into one string with page references.

In [35]:
def format_playbook_context(playbook: list[dict]) -> str:
    """Format playbook chunks with page references for LLM context."""
    # Combine all chunks with page labels and separator lines
    return "\n\n---\n\n".join(
        [f"[Page {c['page']}] {c['content']}" for c in playbook]
    )

Finally, we define the Resolution Agent node. It builds the prompt with the delivery event, the customer profile without the name, locker availability, escalation signals, and playbook sections, then calls the generation model with structured output.

In [36]:
def resolution_agent_node(state: AgentState) -> dict:
    """Resolution Agent: decide whether this is an exception and which action to take."""
    # On a revision, include the Critic's feedback in the prompt
    feedback = ""
    if state["critic_feedback"]:
        feedback = f"\n\nYOUR PREVIOUS DECISION WAS SENT BACK BY THE CRITIC. Feedback:\n{state['critic_feedback']}\nRevise your decision based on this feedback."

    user_content = (
        f"DELIVERY EVENT:\n{json.dumps(state['consolidated_event'], indent=2)}\n\n"
        f"CUSTOMER PROFILE:\n{json.dumps(state['customer_profile'], indent=2)}\n\n"
        f"LOCKER AVAILABILITY:\n{json.dumps(state['locker_availability'], indent=2)}\n\n"
        f"ESCALATION SIGNALS:\n{json.dumps(state['escalation_signals'], indent=2)}\n\n"
        f"RELEVANT PLAYBOOK SECTIONS:\n{format_playbook_context(state['playbook_context'])}"
    )

    result = gen_llm.with_structured_output(ResolutionOutput).invoke([
        SystemMessage(content=RESOLUTION_AGENT_SYSTEM_PROMPT.format(critic_feedback=feedback)),
        HumanMessage(content=user_content)
    ])

    return {
        "resolution_output": result.model_dump(),
        "tool_calls_log": ["AGENT: resolution_agent invoked"],
        "trajectory_log": [f"resolution_agent: is_exception={result.is_exception}, resolution={result.resolution}"]
    }

## Critic Agent

The Critic Agent reviews the Resolution Agent's decision using the validation model. It returns one of three decisions:
- **ACCEPT:** the decision is correct and well reasoned.
- **REVISE:** the decision is wrong or poorly reasoned; the Critic's feedback goes back to the Resolution Agent.
- **ESCALATE:** the decision may be acceptable, but the case needs a human supervisor.

In [37]:
class CriticOutput(BaseModel):
    """Critic Agent output schema."""
    decision: Literal["ACCEPT", "REVISE", "ESCALATE"] = Field(
        description="ACCEPT: valid. REVISE: send back to the Resolution Agent. ESCALATE: needs a human supervisor.")
    rationale: str = Field(description="Reasoning for the review decision")

Next, we define the Critic Agent's system prompt. It lists what to check and includes a firm rule: if the escalation signals contain an AUTOMATIC trigger, the Critic must return ESCALATE.

In [38]:
CRITIC_SYSTEM_PROMPT = """You are the Critic Agent reviewing the Resolution Agent's proposed decision for a delivery event.
Evaluate the Resolution Agent's `is_exception`, `resolution`, and `rationale` against:
- The consolidated delivery event (including `status_code`, `status_description`, `attempt_number`, `package_type`, `package_size`, and `prior_attempt_notes`)
- The customer profile (without PII)
- Locker availability and eligibility constraints
- Escalation signals
- Playbook rules

Return one of three decisions:
- "ACCEPT": The classification and resolution action are accurate, policy-compliant, and well-reasoned.
- "REVISE": The classification, chosen resolution action, or reasoning contradicts the facts, locker eligibility, or playbook rules; provide clear, actionable feedback so the Resolution Agent can fix it.
- "ESCALATE": The case requires human supervisor attention.

Mandatory Rule:
- If `ESCALATION SIGNALS` contains any trigger starting with "AUTOMATIC:", you MUST return "ESCALATE" (unless the decision itself is factually invalid and needs correction first, in which case note the correction clearly while ensuring escalation is triggered)."""

The Critic Agent node sends the full context and the Resolution Agent's decision to the validation model. On REVISE, it stores its feedback and increases the revision count.

In [39]:
def critic_agent_node(state: AgentState) -> dict:
    """Critic Agent: review the Resolution Agent's decision."""
    user_content = (
        f"DELIVERY EVENT:\n{json.dumps(state['consolidated_event'], indent=2)}\n\n"
        f"CUSTOMER PROFILE:\n{json.dumps(state['customer_profile'], indent=2)}\n\n"
        f"LOCKER AVAILABILITY:\n{json.dumps(state['locker_availability'], indent=2)}\n\n"
        f"ESCALATION SIGNALS:\n{json.dumps(state['escalation_signals'], indent=2)}\n\n"
        f"PLAYBOOK CONTEXT:\n{format_playbook_context(state['playbook_context'])}\n\n"
        f"RESOLUTION AGENT OUTPUT:\n{json.dumps(state['resolution_output'], indent=2)}"
    )

    result = eval_llm.with_structured_output(CriticOutput).invoke([
        SystemMessage(content=CRITIC_SYSTEM_PROMPT),
        HumanMessage(content=user_content)
    ])

    update = {
        "critic_output": result.model_dump(),
        "tool_calls_log": ["AGENT: critic_agent invoked"],
        "trajectory_log": [f"critic_agent: decision={result.decision}"]
    }
    # On REVISE, pass the feedback to the Resolution Agent's next attempt
    if result.decision == "REVISE":
        update["critic_feedback"] = result.rationale
        update["revision_count"] = state["revision_count"] + 1
    return update

## Communication Agent

We first define the output schema for the Communication Agent: a tone label and the customer message.

In [40]:
class CommunicationOutput(BaseModel):
    """Communication Agent output schema."""
    tone_label: Literal["FORMAL", "CASUAL"] = Field(description="Tone of the customer message, based on customer tier")
    communication_message: str = Field(description="The customer-facing notification message")

Next, we define the Communication Agent's system prompt, which sets tone rules by customer tier, the required message content, channel formatting (EMAIL or SMS), service credit acknowledgment, and locker reroute details.

In [41]:
COMMUNICATION_AGENT_SYSTEM_PROMPT = """You are the Communication Agent responsible for writing personalized customer notifications for delivery exceptions.

Follow these rules:
1. Tone by customer tier:
   - Set `tone_label` to "FORMAL" for VIP or high-touch/formal cases, and "CASUAL" for standard/everyday updates as appropriate to customer tier and context (matching playbook guidelines).
2. Channel adaptation:
   - Format the message appropriately for `preferred_channel` ("SMS" should be concise and direct; "EMAIL" can include fuller context, subject/greeting, and clear next steps).
3. Required details:
   - Address the customer politely using `customer_name`.
   - Clearly explain the delivery status/exception (`exception_type`, `package_type`, and relevant context) and the chosen `resolution` action.
   - If `active_credit` is greater than 0, acknowledge their available account/service credit smoothly.
   - If `resolution` is "REROUTE_TO_LOCKER" and `locker_for_reroute` is provided, include the locker ID, address, and operating hours clearly."""

The Communication Agent node is the only agent that reads the customer's name. It sends the generation model the resolution, the customer's tier, channel, and credit, the exception details, and, for locker reroutes, the first eligible locker.

In [42]:
def communication_agent_node(state: AgentState) -> dict:
    """Communication Agent: write the customer notification."""
    event = state["consolidated_event"]
    profile = state["customer_profile_full"]
    resolution = state["resolution_output"]

    context = {
        "customer_name": profile.get("name", "Customer"),
        "customer_tier": profile.get("tier"),
        "preferred_channel": profile.get("preferred_channel"),
        "active_credit": profile.get("active_credit", 0),
        "exception_type": event["status_code"],
        "status_description": event["status_description"],
        "package_type": event["package_type"],
        "resolution": resolution.get("resolution"),
        "resolution_rationale": resolution.get("rationale")
    }

    # For locker reroutes, include the first eligible locker's details
    eligible = [l for l in state["locker_availability"] if l.get("eligible")]
    if resolution.get("resolution") == "REROUTE_TO_LOCKER" and eligible:
        context["locker_for_reroute"] = eligible[0]

    result = gen_llm.with_structured_output(CommunicationOutput).invoke([
        SystemMessage(content=COMMUNICATION_AGENT_SYSTEM_PROMPT),
        HumanMessage(content=f"CONTEXT:\n{json.dumps(context, indent=2)}")
    ])

    return {
        "communication_output": result.model_dump(),
        "tool_calls_log": ["AGENT: communication_agent invoked"],
        "trajectory_log": [f"communication_agent: tone={result.tone_label}"]
    }

## Finalize

The finalize node packages the results into one output dictionary and records the end-to-end latency.

In [43]:
def finalize_node(state: AgentState) -> dict:
    """Package the final output and record latency."""
    final = {
        "shipment_id": state["shipment_id"],
        "is_exception": state["resolution_output"].get("is_exception", "ERROR"),
        "resolution": state["resolution_output"].get("resolution", "ERROR"),
        "escalated": state["escalated"],
        "tone": state["communication_output"].get("tone_label", "N/A"),
        "message": state["communication_output"].get("communication_message", ""),
        "revision_count": state["revision_count"]
    }
    latency = time.time() - state["start_time"]
    return {
        "final_output": final,
        "latency_sec": latency,
        "trajectory_log": [f"finalize: actions={json.dumps(final)}; latency={latency:.3f}s"]
    }

# **Multi-Agent Workflow**

We now connect the nodes into a LangGraph workflow.
- The preprocessor and every agent report back to the supervisor with a fixed edge.
- The supervisor uses a conditional edge to send the case to the node named in `next_agent`.
- Finalize ends the workflow.

In [ ]:
# write the code to define the LangGraph workflow and compile it into an app named app

Let's visualize the workflow.

In [ ]:
# write the code to visualize the LangGraph workflow

# **Evaluation Metrics**

We now define the evaluation metrics for the multi-agent system.

## Task Completion Rate

This function computes task completion for a single shipment by checking three sub-criteria:
- whether the exception classification matches ground truth,
- whether the resolution action is correct, and
- whether the communication tone is appropriate.

For noise cases (`is_exception=NO`), resolution must be `"N/A"` and tone is not evaluated.

A task is complete ONLY when ALL three sub-criteria pass.

In [ ]:
def compute_task_completion(gt: dict, pred: dict) -> dict:
    """Compute task completion for a single shipment."""
    res = pred.get("resolution_output", {})
    comm = pred.get("communication_output", {})

    # Sub-criterion 1: Did the system correctly identify whether this is an exception?
    exception_correct = gt.get("is_exception") == res.get("is_exception")

    if gt.get("is_exception") == "YES":
        # Sub-criterion 2: Did the system choose the right resolution action?
        resolution_correct = gt.get("expected_resolution") == res.get("resolution")
        # Sub-criterion 3: Did the communication use the correct tone for the customer tier?
        tone_correct = gt.get("expected_tone") == comm.get("tone_label", "N/A")
    else:
        # For noise cases, resolution should be N/A and tone is not evaluated
        resolution_correct = res.get("resolution") in ("N/A", None, "")
        tone_correct = True

    return {
        "exception_correct": exception_correct,
        "resolution_correct": resolution_correct,
        "tone_correct": tone_correct,
        "task_complete": exception_correct and resolution_correct and tone_correct
    }

## Escalation Accuracy

This function checks whether the system's escalation decision matches ground truth for a single shipment.
- Noise cases, where the ground truth is `"N/A"`, are excluded from evaluation by returning `None`.
- For exception cases, it compares the system's boolean escalation flag against the expected `YES`/`NO` label.

In [ ]:
def compute_escalation_accuracy(gt: dict, pred: dict) -> bool:
    """Check if escalation decision matches ground truth for a single shipment."""
    # Noise cases have no escalation expectation - skip evaluation
    if gt.get("should_escalate") not in ("YES", "NO"):
        return None

    # Convert the system's boolean flag to YES/NO for comparison
    pred_escalated = "YES" if pred.get("escalated", False) else "NO"

    # Compare against ground truth
    return gt.get("should_escalate") == pred_escalated

## Tool Call Accuracy

This function verifies that the correct tools and agents were invoked for a single shipment based on its type. Three paths are checked:
- routine noise cases (flagged by the preprocessor) should have no tool calls or agent invocations, since they skip all processing,
- exception cases must invoke the four context tools, the Resolution Agent, and the Communication Agent, and
- non-exception cases that aren't routine noise must invoke the four context tools and the Resolution Agent, but not the Communication Agent.

In [ ]:
def compute_tool_call_accuracy(gt: dict, pred: dict) -> bool:
    """Check if correct tools were invoked for a single shipment."""
    # Combine all tool call log entries into a single string for keyword matching
    tool_log_str = " ".join(pred.get("tool_calls_log", []))
    is_exception = gt.get("is_exception", "NO")
    is_noise = pred.get("is_noise", False)

    if is_noise:
        # Routine noise skips all tools and agents - verify nothing was invoked
        skip_tools = ["lookup_customer_profile", "check_locker_availability",
                      "search_playbook", "check_escalation_rules",
                      "resolution_agent", "communication_agent"]
        return not any(t in tool_log_str for t in skip_tools)

    # Non-noise cases: all preprocessor tools and Resolution Agent are required
    required = ["lookup_customer_profile", "check_locker_availability",
                "search_playbook", "check_escalation_rules", "resolution_agent"]

    # Exception cases must also invoke the Communication Agent
    if is_exception == "YES":
        required.append("communication_agent")

    all_present = all(t in tool_log_str for t in required)

    # Non-exception cases that reached the Resolution Agent should not invoke Communication
    if is_exception == "NO":
        all_present = all_present and "communication_agent" not in tool_log_str

    return all_present

## Reasoning Trajectory Coherence

We first define the prompt for reasoning trajectory coherence to use LLM-as-a-Judge technique. It
- receives the full pipeline trace for a single shipment, including all agent outputs, the Critic's decision, the revision count, and the trajectory log, and
- scores coherence on a 1-5 scale based on whether each agent's reasoning logically follows from its inputs with no contradictions.

In [ ]:
REASONING_TRAJECTORY_COHERENCE_PROMPT = """

-----WRITE THE PROMPT HERE-----

Ask the judge to respond with ONLY a JSON object: {"score": <1-5>, "justification": "<brief explanation>"}

"""

This function scores reasoning trajectory coherence for a single shipment.
- Assembles a trace with all agent outputs, the Critic's decision, the revision count, and the trajectory log, with no customer names sent to the evaluator.
- Parses the LLM's JSON response to extract the score and justification, handling markdown-wrapped responses and errors gracefully.

In [ ]:
def compute_coherence_score(pred: dict) -> dict:
    """Use eval_llm (the validation model) to score reasoning trajectory coherence."""
    # Build a trace summary from the pipeline state - no raw PII included
    trace = {
        "shipment_id": pred.get("shipment_id"),
        "consolidated_event": pred.get("consolidated_event", {}),
        "customer_tier": pred.get("customer_profile", {}).get("tier"),
        "escalation_signals": pred.get("escalation_signals", {}),
        "resolution_output": pred.get("resolution_output", {}),
        "critic_output": pred.get("critic_output", {}),
        "revision_count": pred.get("revision_count", 0),
        "communication_output": pred.get("communication_output", {}),
        "trajectory_log": pred.get("trajectory_log", [])
    }

    try:
        # Invoke the evaluation judge
        response = eval_llm.invoke([
            SystemMessage(content=REASONING_TRAJECTORY_COHERENCE_PROMPT),
            HumanMessage(content=json.dumps(trace, indent=2))
        ])

        # Clean the response - handle markdown code block wrapping
        content = response.content.strip()
        if content.startswith("```"):
            content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        # Parse the JSON response
        parsed = json.loads(content)
        return {
            "score": parsed.get("score", 0),
            "justification": parsed.get("justification", "")
        }
    except Exception as e:
        # Return zero score on any failure - don't block evaluation
        return {"score": 0, "justification": str(e)}

## Helper Function for Single Test Case Metrics

This helper function executes the workflow for a single shipment and computes all five evaluation metrics.
- Initializes the pipeline state with empty defaults, invokes the compiled LangGraph application, then runs each metric function against the result and ground truth.
- Also extracts playbook page citations for the test case output.

In [ ]:
def run_single_test_case(shipment_id: str, rows: list[dict], gt: dict) -> dict:
    """Run the workflow for one shipment and compute all five metrics."""
    # Start every shipment from the same empty state
    initial_state = {
        "shipment_id": shipment_id,
        "raw_rows": rows,
        "consolidated_event": {},
        "is_noise": False,
        "customer_profile": {},
        "customer_profile_full": {},
        "locker_availability": [],
        "playbook_context": [],
        "escalation_signals": {},
        "resolution_output": {},
        "critic_output": {},
        "critic_feedback": "",
        "revision_count": 0,
        "communication_output": {},
        "next_agent": "",
        "escalated": False,
        "tool_calls_log": [],
        "trajectory_log": [],
        "start_time": 0.0,
        "latency_sec": 0.0,
        "final_output": {}
    }

    # Run the full workflow
    result = app.invoke(initial_state)

    # Compute the metrics for this shipment
    citations = [f"Page {c['page']}" for c in result.get("playbook_context", [])]
    return {
        "state": result,                                       # Full final state, for inspection
        "task_completion": compute_task_completion(gt, result),
        "escalation_correct": compute_escalation_accuracy(gt, result),
        "tool_call_correct": compute_tool_call_accuracy(gt, result),
        "coherence": compute_coherence_score(result),
        "latency": result.get("latency_sec", 0.0),
        "citations": citations                                 # Playbook pages retrieved
    }

# **Test Cases**

## Prepare Ground Truth

We prepare the test case inputs.
- Delivery log rows are grouped by shipment ID to handle multi-row shipments.
- Ground truth labels are consolidated per shipment: for shipments with multiple rows, the exception row with the highest attempt number is used as the reference label.
- Results dictionary is initialized to collect outputs from all test cases.

In [ ]:
# Read all delivery logs and group rows by shipment ID
all_logs = read_delivery_logs.invoke({})
shipment_groups = defaultdict(list)
for row in all_logs:
    shipment_groups[row["shipment_id"]].append(row)

# Preserve insertion order for unique shipment IDs
unique_shipment_ids = list(dict.fromkeys(row["shipment_id"] for row in all_logs))

# Build consolidated ground truth - one label per shipment
gt_by_shipment = defaultdict(list)
for _, row in ground_truth_df.iterrows():
    gt_by_shipment[row["shipment_id"]].append(row.to_dict())

# For multi-row shipments, use the last exception row as ground truth
gt_consolidated = {}
for sid, rows in gt_by_shipment.items():
    exc_rows = [r for r in rows if r["is_exception"] == "YES"]
    gt_consolidated[sid] = exc_rows[-1] if exc_rows else rows[0]

print(f"Shipments to process: {unique_shipment_ids}")

# Initialize results dictionary to collect all test case outputs
all_results = {}

## Helper Function for Test Case Runs

This helper function prints the complete output for a single test case
- predictions versus ground truth,
- all five evaluation metrics,
- the full agent trajectory trace,
- playbook page citations, and
- a preview of the customer message if one was generated.

In [ ]:
def print_test_case_output(sid: str, result: dict, gt: dict):
    """Pretty-print a single test case result."""
    state = result["state"]
    res = state.get("resolution_output", {})
    comm = state.get("communication_output", {})
    tc = result["task_completion"]

    # --- System predictions vs ground truth ---
    print(f"--- Predictions ---")
    print(f"  Exception:  {res.get('is_exception', 'N/A')} (GT: {gt.get('is_exception')})")
    print(f"  Resolution: {res.get('resolution', 'N/A')} (GT: {gt.get('expected_resolution')})")
    pred_esc = 'YES' if state.get('escalated') else 'NO'
    print(f"  Escalated:  {pred_esc} (GT: {gt.get('should_escalate')})")
    print(f"  Tone:       {comm.get('tone_label', 'N/A')} (GT: {gt.get('expected_tone')})")
    print(f"  Revisions:  {state.get('revision_count', 0)}")

    # --- Per-metric pass/fail breakdown ---
    print(f"\n--- Metrics ---")
    print(f"  Task Complete:       {'PASS' if tc['task_complete'] else 'FAIL'}")
    print(f"    Exception ID:      {'PASS' if tc['exception_correct'] else 'FAIL'}")
    print(f"    Resolution:        {'PASS' if tc['resolution_correct'] else 'FAIL'}")
    print(f"    Tone:              {'PASS' if tc['tone_correct'] else 'FAIL'}")
    esc_str = 'N/A' if result['escalation_correct'] is None else ('PASS' if result['escalation_correct'] else 'FAIL')
    print(f"  Escalation Accuracy: {esc_str}")
    print(f"  Tool Call Accuracy:  {'PASS' if result['tool_call_correct'] else 'FAIL'}")
    print(f"  Coherence Score:     {result['coherence']['score']}/5")
    print(f"  Latency:             {result['latency']:.2f}s")

    # --- Full agent decision trail ---
    print(f"\n--- Trajectory ---")
    for entry in state.get("trajectory_log", []):
        print(f"  {entry}")

    # --- Which playbook pages were retrieved ---
    print(f"\n--- Document Citations ---")
    print(f"  Playbook pages referenced: {', '.join(result['citations']) if result['citations'] else 'None'}")

    # --- Preview of the customer notification if generated ---
    if comm.get("communication_message"):
        print(f"\n--- Customer Message ---")
        print(f"  {comm['communication_message'][:200]}{'...' if len(comm.get('communication_message','')) > 200 else ''}")

## Test Case 1: SHP-001 - Successful Delivery (Noise Filtering)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 2: SHP-002 - VIP Multi-Attempt with Duplicate Scan

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 3: SHP-003 - Address Not Found (Standard Customer)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 4: SHP-004 - Damaged Fragile Package (VIP Escalation)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 5: SHP-005 - Third Failed Attempt (Locker Full, Mandatory Escalation)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 6: SHP-006 - Refused Delivery (Return to Sender)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 7: SHP-007 - Perishable Weather Delay Exceeding Threshold (VIP Replace)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 8: SHP-008 - Discretionary Escalation (High Exception History)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 9: SHP-009 - Severely Damaged Perishable (Premium Replace)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

## Test Case 10: SHP-010 - Routine Depot Scan (Noise Filtering)

In [ ]:
# write the code to send the inputs to the multi-agent system and print the response, traces, and evaluation metrics

# **Aggregated Evaluation Metrics**

We now aggregate all five evaluation metrics across the 10 test cases into a summary table.

- Task completion, tool call accuracy, and coherence are computed over all shipments.
- Escalation accuracy excludes noise cases where ground truth is `"N/A"`.
- Latency is reported as the mean across all runs.

In [ ]:
# write the code to print the aggregate metrics across all test cases

We print the detailed per-shipment results showing ground truth versus predictions for each evaluation dimension, along with pass/fail status for task completion and tool calls, the coherence score, and latency.

This provides a complete at-a-glance view of where the system succeeded and where it diverged.

In [ ]:
# write the code to print the results for all test cases

# **Conclusions and Business Recommendations**

## Conclusions

- A
- B
- C

## Business Recommendations

- A
- B
- C

<font size=6 color='#4682B4'>Power Ahead</font>
___